# Qwen3.5-27B Q&A Smoke Test

This notebook tests `Qwen/Qwen3.5-27B` with tiny Q&A prompts outside the extraction evaluator.

Goal: determine whether gibberish output comes from the model/runtime/load path or from the extraction prompt.

In [ ]:
import os
import platform
import torch
import transformers

print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES'))

if torch.cuda.is_available():
    print('CUDA device count:', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f'cuda:{i}: {props.name}, {props.total_memory / 1024**3:.1f} GiB')

## Configuration

On a cluster with one allocated GPU, `cuda:0` means the first GPU visible inside the job.

In [ ]:
MODEL_ID = 'Qwen/Qwen3.5-27B'
DEVICE = 'auto'       # Use 'auto', 'cuda:0', or 'cpu'
LOAD_IN_4BIT = False  # Set True only if bitsandbytes is installed and you need quantization
MAX_NEW_TOKENS = 128

GENERATION_KWARGS = {
    'max_new_tokens': MAX_NEW_TOKENS,
    'do_sample': False,
    'repetition_penalty': 1.0,
}

print('MODEL_ID:', MODEL_ID)
print('DEVICE:', DEVICE)
print('LOAD_IN_4BIT:', LOAD_IN_4BIT)
print('GENERATION_KWARGS:', GENERATION_KWARGS)

## Load Model

This uses the Qwen3.5 model-card style loader: `AutoProcessor` plus `AutoModelForImageTextToText`.

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor

model_kwargs = {'dtype': 'auto'}

if DEVICE == 'auto':
    model_kwargs['device_map'] = 'auto'
elif LOAD_IN_4BIT:
    model_kwargs['device_map'] = {'': DEVICE}

if LOAD_IN_4BIT:
    from transformers import BitsAndBytesConfig
    model_kwargs['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True)

print('Loading processor...')
processor = AutoProcessor.from_pretrained(MODEL_ID)

print('Loading model...')
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, **model_kwargs)

if DEVICE != 'auto' and not LOAD_IN_4BIT:
    model.to(torch.device(DEVICE))

model.eval()

print('Model class:', type(model).__name__)
print('Processor class:', type(processor).__name__)
print('Model dtype:', getattr(model, 'dtype', None))
print('Model device:', getattr(model, 'device', None))
print('HF device map:', getattr(model, 'hf_device_map', None))
print('Generation config:', model.generation_config)

In [ ]:
def first_input_device(model):
    device_map = getattr(model, 'hf_device_map', None)
    if device_map:
        for value in device_map.values():
            value = str(value)
            if value not in {'cpu', 'disk'} and not value.startswith('cpu'):
                return torch.device(value)
    return getattr(model, 'device', torch.device('cpu'))

INPUT_DEVICE = first_input_device(model)
print('Input device:', INPUT_DEVICE)

## Q&A Helper

In [ ]:
def run_qa(question, *, enable_thinking=False, system='You are a concise helpful assistant.'):
    messages = [
        {'role': 'system', 'content': [{'type': 'text', 'text': system}]},
        {'role': 'user', 'content': [{'type': 'text', 'text': question}]},
    ]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        return_tensors='pt',
        add_generation_prompt=True,
        enable_thinking=enable_thinking,
    )
    inputs = inputs.to(INPUT_DEVICE)

    input_len = inputs['input_ids'].shape[-1]
    with torch.inference_mode():
        output_ids = model.generate(**inputs, **GENERATION_KWARGS)

    new_ids = output_ids[0][input_len:]
    raw = processor.decode(new_ids, skip_special_tokens=False)
    clean = processor.decode(new_ids, skip_special_tokens=True)

    return {
        'question': question,
        'enable_thinking': enable_thinking,
        'input_tokens': int(input_len),
        'new_tokens': int(new_ids.shape[-1]),
        'new_token_ids_head': new_ids[:50].tolist(),
        'raw': raw,
        'clean': clean,
    }

def print_result(result):
    print('=' * 100)
    print('Question:', result['question'])
    print('Thinking:', result['enable_thinking'])
    print('Input tokens:', result['input_tokens'])
    print('New tokens:', result['new_tokens'])
    print('First generated token IDs:', result['new_token_ids_head'])
    print('\nRAW OUTPUT:')
    print(result['raw'])
    print('\nCLEAN OUTPUT:')
    print(result['clean'])

## Non-Thinking Smoke Tests

These should be simple and stable. If these produce gibberish, focus on the runtime/model loading path.

In [ ]:
tests = [
    'What is 2 + 2? Answer with only the number.',
    'What is the capital of France? Answer in one short sentence.',
    'Return exactly this JSON: {"answer": "ok"}',
]

for question in tests:
    print_result(run_qa(question, enable_thinking=False))

## Optional Thinking-Mode Comparison

Run this only after non-thinking works. Thinking mode can be much slower.

In [ ]:
print_result(run_qa('What is 2 + 2? Answer with only the number.', enable_thinking=True))

## Interpretation

- If non-thinking Q&A is gibberish, the evaluator is not the culprit. Check Transformers version, PyTorch/CUDA compatibility, quantization, sharding, and checkpoint cache integrity.
- If non-thinking Q&A works but thinking mode fails or is too slow, use `--qwen_thinking off` in the evaluator.
- If this notebook works but the evaluator fails, compare prompt length, generation settings, and parsing retries.